<a href="https://colab.research.google.com/github/Kamilr616/AI_sign_language_translator/blob/develop/notebooks/Custom_gesture_recognizer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Project: /mediapipe/_project.yaml
Book: /mediapipe/_book.yaml


# Hand gesture recognition model customization

The MediaPipe Model Maker package is a low-code solution for customizing on-device machine learning (ML) Models.

This notebook shows the end-to-end process of customizing a gesture recognizer model for recognizing ASL alphabet signs.

## Prerequisites

Install the MediaPipe Model Maker package.

In [2]:
!pip install --upgrade pip
#TODO: ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts. inflect 7.3.1 requires typeguard>=4.0.1, but you have typeguard 2.13.3 which is incompatible.
!pip install mediapipe-model-maker
!pip install kaggle

  Using cached pip-24.2-py3-none-any.whl.metadata (3.6 kB)
Using cached pip-24.2-py3-none-any.whl (1.8 MB)
  Attempting uninstall: pip
    Found existing installation: pip 24.1.2
    Uninstalling pip-24.1.2:
      Successfully uninstalled pip-24.1.2
  Preparing metadata (setup.py) ... done
INFO: pip is looking at multiple versions of tf-keras to determine which version is compatible with other requirements. This could take a while.
INFO: pip is looking at multiple versions of tensorflow-metadata to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.9/35.9 MB 70.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 475.2/475.2 MB 27.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.7/2.7 MB 86.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.2/5.2 MB 28.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 611.8/611.8 kB 18.2 MB/s eta 0:00:00
   ━━━━━━━━━━

Import the required libraries.

In [3]:
from google.colab import files
import os
import tensorflow as tf
import matplotlib.pyplot as plt

assert tf.__version__.startswith('2')

from mediapipe_model_maker import gesture_recognizer

/usr/local/lib/python3.10/dist-packages/tensorflow_addons/utils/tfa_eol_msg.py:23: UserWarning: 

TensorFlow Addons (TFA) has ended development and introduction of new features.
TFA has entered a minimal maintenance and release mode until a planned end of life in May 2024.
Please modify downstream libraries to take dependencies from other repositories in our TensorFlow community (e.g. Keras, Keras-CV, and Keras-NLP). 

For more information see: https://github.com/tensorflow/addons/issues/2807 

  warnings.warn(


### Get the dataset

The dataset for gesture recognition in model maker requires the following format: `<dataset_path>/<label_name>/<img_name>.*`. In addition, one of the label names (`label_names`) must be `none`. The `none` label represents any gesture that isn't classified as one of the other gestures.

This code fetches data from kaggle directly. kaggle.json file is required.

In [5]:
files.upload() #kaggle.json
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

import kaggle

kaggle_dataset_name = "grassknoted/asl-alphabet"
try:
  kaggle.api.dataset_download_files(kaggle_dataset_name, path='datasets/', unzip=True)
  print(f"Successfully downloaded {kaggle_dataset_name}")
  dataset_train_path = "datasets/asl_alphabet_train/asl_alphabet_train"
  os.rename(f"{dataset_train_path}/nothing", f"{dataset_train_path}/none")
except Exception as e:
  print(f"Failed to load {kaggle_dataset_name}: {e}")


Saving kaggle.json to kaggle.json
Dataset URL: https://www.kaggle.com/datasets/grassknoted/asl-alphabet
Successfully downloaded grassknoted/asl-alphabet


Verify the dataset by printing the labels. There should be all gesture labels, with one of them being the `none` gesture.

In [6]:
print(dataset_train_path)
labels = []
for i in os.listdir(dataset_train_path):
  if os.path.isdir(os.path.join(dataset_train_path, i)):
    labels.append(i)
print(labels)

datasets/asl_alphabet_train/asl_alphabet_train
['A', 'U', 'H', 'O', 'N', 'I', 'B', 'L', 'Q', 'D', 'P', 'none', 'K', 'del', 'V', 'G', 'J', 'F', 'W', 'space', 'E', 'Z', 'X', 'Y', 'M', 'T', 'S', 'C', 'R']


### Create the model
The workflow consists of 4 steps which have been separated into their own code blocks.

**Load the dataset**

Load the dataset located at `dataset_path` by using the `Dataset.from_folder` method. When loading the dataset, run the pre-packaged hand detection model from MediaPipe Hands to detect the hand landmarks from the images. Any images without detected hands are ommitted from the dataset. The resulting dataset will contain the extracted hand landmark positions from each image, rather than images themselves.

The `HandDataPreprocessingParams` class contains two configurable options for the data loading process:
* `shuffle`: A boolean controlling whether to shuffle the dataset. Defaults to true.
* `min_detection_confidence`: A float between 0 and 1 controlling the confidence threshold for hand detection. (60%)

Split the dataset: 80% for training, 18% for validation, and 2% for testing.

In [8]:
data = gesture_recognizer.Dataset.from_folder(
    dirname=dataset_train_path,
    hparams=gesture_recognizer.HandDataPreprocessingParams(shuffle=True, min_detection_confidence=0.6)
)
train_data, rest_data = data.split(0.8)
validation_data, test_data = rest_data.split(0.9)

/usr/local/lib/python3.10/dist-packages/google/protobuf/symbol_database.py:55: UserWarning: SymbolDatabase.GetPrototype() is deprecated. Please use message_factory.GetMessageClass() instead. SymbolDatabase.GetPrototype() will be removed soon.
  warnings.warn('SymbolDatabase.GetPrototype() is deprecated. Please '


In [9]:
# TODO: save dataset to file
# import numpy
# import pickle

# tf_dataset = data.gen_tf_dataset()
# numpy_images = []
# numpy_labels = []

# for images, labels in tf_dataset:
#     numpy_images.append(images.numpy())
#     numpy_labels.append(labels.numpy())

# with open('asl_gesture_dataset.pkl', 'wb') as f:
#     pickle.dump((numpy_images, numpy_labels), f)

# files.download('asl_gesture_dataset.pkl')

In [10]:
# TODO: read dataset from file
# import pickle

# files.upload()
# # Załaduj dataset z pliku
# with open('asl_gesture_dataset.pkl', 'rb') as f:
#     numpy_images, numpy_labels = pickle.load(f)

# def remove_extra_dimension(image):
#     image = tf.squeeze(image)
#     return image

# tf_data = tf.data.Dataset.from_tensor_slices((numpy_images))

# tf_data = tf_data.map(remove_extra_dimension)

# data = gesture_recognizer.Dataset(dataset=tf_data, label_names=list(numpy_labels), size=59707)

# train_data, rest_data = data.split(0.8)
# validation_data, test_data = rest_data.split(0.9)

## Hyperparameters {:#hyperparameters}


You can further customize the model using the `GestureRecognizerOptions` class, which has two optional parameters for `ModelOptions` and `HParams`. Use the `ModelOptions` class to customize parameters related to the model itself, and the `HParams` class to customize other parameters related to training and saving the model.

`ModelOptions` has one customizable parameter that affects accuracy:
* `dropout_rate`: The fraction of the input units to drop. Used in dropout layer. Defaults to 0.05.
* `layer_widths`: A list of hidden layer widths for the gesture model. Each element in the list will create a new hidden layer with the specified width. The hidden layers are separated with BatchNorm, Dropout, and ReLU. Defaults to an empty list(no hidden layers).

`HParams` has the following list of customizable parameters which affect model accuracy:
* `learning_rate`: The learning rate to use for gradient descent training. Defaults to 0.001.
* `batch_size`: Batch size for training. Defaults to 2.
* `epochs`: Number of training iterations over the dataset. Defaults to 10.
* `steps_per_epoch`: An optional integer that indicates the number of training steps per epoch. If not set, the training pipeline calculates the default steps per epoch as the training dataset size divided by batch size.
* `shuffle`: True if the dataset is shuffled before training. Defaults to False.
* `lr_decay`: Learning rate decay to use for gradient descent training. Defaults to 0.99.
* `gamma`: Gamma parameter for focal loss. Defaults to 2

Additional `HParams` parameter that does not affect model accuracy:
* `export_dir`: The location of the model checkpoint files and exported model files.

Set manual Hyperparameters

In [21]:
_dropout_rate = 0.075
_layer_width = [128, 64]
_learning_rate = 0.001
_batch_size = 16
_epochs = 60
_steps_per_epoch = None
_shuffle = True
_lr_decay = 0.9
_gamma = 2

**Train the model**

Train the custom gesture recognizer by using the create method and passing in the training data, validation data, model options, and hyperparameters. For more information on model options and hyperparameters, see the [Hyperparameters](#hyperparameters) section.

In [ ]:
hparams = gesture_recognizer.HParams(learning_rate=_learning_rate, export_dir="exported_model", batch_size=_batch_size, epochs=_epochs, steps_per_epoch=_steps_per_epoch, shuffle=_shuffle, lr_decay=_lr_decay, gamma=_gamma)

model_options = gesture_recognizer.ModelOptions(dropout_rate=_dropout_rate, layer_widths=_layer_width)
options = gesture_recognizer.GestureRecognizerOptions(model_options=model_options, hparams=hparams)
model = gesture_recognizer.GestureRecognizer.create(
    train_data=train_data,
    validation_data=validation_data,
    options=options
)

Model: "model_4"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 hand_embedding (InputLayer  [(None, 128)]             0         
 )                                                               
                                                                 
 batch_normalization_13 (Ba  (None, 128)               512       
 tchNormalization)                                               
                                                                 
 re_lu_13 (ReLU)             (None, 128)               0         
                                                                 
 dropout_13 (Dropout)        (None, 128)               0         
                                                                 
 custom_gesture_recognizer_  (None, 128)               16512     
 0 (Dense)                                                       
                                                           

**Evaluate the model performance**

After training the model, evaluate it on a test dataset and print the loss and accuracy metrics.

In [17]:
loss, accuracy = model.evaluate(test_data)
print(f"Test loss:{loss}, Test accuracy:{accuracy}")

75/75 [==============================] - 29s 17ms/step - loss: 0.0424 - categorical_accuracy: 0.9711
Test loss:0.042438142001628876, Test accuracy:0.9711176156997681


**Export to Tensorflow Lite Model**

After creating the model, convert and export it to a Tensorflow Lite model format for later use on an on-device application. The export also includes model metadata, which includes the label file.

In [18]:
model.export_model()
files.download('exported_model/gesture_recognizer.task')


Using existing files at /tmp/model_maker/gesture_recognizer/palm_detection_full.tflite
Using existing files at /tmp/model_maker/gesture_recognizer/hand_landmark_full.tflite


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>